# Alpine and Icelandic Dashboard

This notebook is self-contained and does not rely on OGGM bindings.

In [ ]:
# Install dependencies (run once)
# !pip install panel holoviews bokeh param xarray geopandas -q

# !uv pip install tables

In [ ]:
import asyncio
from pathlib import Path
from typing import Dict, Optional
import time
import json
import panel as pn
import holoviews as hv
import dtcg.interface.plotting as dtcg_plotting
import numpy as np
import xarray as xr

import geopandas as gpd
from datetime import date

# Configure panel
pn.extension("notifications")
pn.extension(design="material", sizing_mode="stretch_width")

print("Dependencies loaded")

## 1. Data Layer

In [ ]:
class GlacierDataCache:
    """Handle all data loading and caching."""

    def __init__(self, cache_path: str = "./static/data/l2_precompute/"):
        self.cache_path = Path(cache_path)
        self._glacier_cache: Dict[str, dict] = {}  # Local memory cache
        self._glacier_index: Optional[dict] = None

    def get_glacier_index(self) -> dict:
        """Get glacier_index."""
        if self._glacier_index is None:
            # In real implementation: Load from glacier_index.json
            self._glacier_index = {
                "Central Europe": {
                    "Hintereisferner": "RGI60-11.00897",
                    "Kesselwandferner": "RGI60-11.00787",
                    "Vernagtferner": "RGI60-11.00719",
                },
                "Iceland": {
                    "Bruarjoekull": "RGI60-06.00377",
                    "Skeidararjoekull": "RGI60-06.00475",
                    "Breidamerkurjoekull": "RGI60-06.00483",
                },
            }
            # self._glacier_index = self.get_cached_glacier_index(cache=self.cache_path)
        return self._glacier_index

    async def get_glacier_data(self, rgi_id: str) -> dict:
        """Get data for a single glacier."""

        # Check if already cached
        if rgi_id not in self._glacier_cache:
            data = self.get_cached_data(rgi_id=rgi_id)
            self._glacier_cache[rgi_id] = data
        return self._glacier_cache[rgi_id]

    def get_cached_data(self, rgi_id: str) -> dict:
        """Get precomputed data for a single glacier.

        Parameters
        ----------
        rgi_id : str
            Glacier RGI ID.

        Returns
        -------
        dict
            Cached glacier dataset containing minimal GlacierDirectory
            data, datacube, time series for specific mass balance and
            runoff, and glacier outline. Unavailable datasets default
            to a ``NoneType`` object.
        """
        cached_data = self.get_cached_datasets(rgi_id=rgi_id, cache=self.cache_path)
        data = {
            "gdir": cached_data.get("gdir", None),
            "eolis": cached_data.get("eolis", None),
            "smb": cached_data.get("smb", None),
            "runoff_data": cached_data.get("runoff", None),
            "outlines": cached_data.get("outlines", None),
        }

        return data

    def get_cached_datasets(
        self, rgi_id: str, cache="./static/data/l2_precompute/"
    ) -> dict:

        if isinstance(cache, str):
            cache = Path(cache)
        cache_path = cache / rgi_id
        gdir = self.get_cached_gdir_data(cache_path=cache_path)
        smb = self.get_cached_smb_data(cache_path=cache_path)
        runoff = {}
        for key in [
            "Daily_Hugonnet_2000_2020",
            "Daily_Cryosat_2011_2020",
            "SfcDaily_Cryosat_2011_2020",
        ]:
            runoff[key] = self.get_cached_runoff_data(cache_path=cache_path, suffix=key)
        eolis = self.get_cached_eolis_data(cache_path=cache_path)
        outlines = self.get_cached_outline_data(cache_path=cache_path)

        cached_data = {
            "gdir": gdir,
            "smb": smb,
            "runoff": runoff,
            "eolis": eolis,
            "outlines": outlines,
        }

        return cached_data

    def get_cached_gdir_data(self, cache_path: Path) -> dict:

        try:
            with open(cache_path / "gdir.json", mode="r", encoding="utf-8") as file:
                raw = file.read()
                gdir = dict(json.loads(raw))
        except FileNotFoundError:
            return None
        return gdir

    def get_cached_smb_data(self, cache_path: Path) -> np.ndarray:
        try:
            smb = np.load(cache_path / "smb.npz")
        except FileNotFoundError:
            return None

        return smb

    def get_cached_runoff_data(
        self, cache_path: Path, suffix="Daily_Hugonnet_2000_2020"
    ) -> dict:
        try:
            runoff = xr.open_dataarray(cache_path / f"runoff_{suffix}.nc")
        except FileNotFoundError:
            return None

        runoff_data = {"monthly_runoff": runoff}
        return runoff_data

    def get_cached_eolis_data(self, cache_path: Path) -> xr.DataArray:
        try:
            eolis_data = xr.open_dataset(cache_path / "eolis.nc")
        except FileNotFoundError:
            return None
        return eolis_data

    def get_cached_glacier_index(
        self, index="glacier_index", cache="./static/data/l2_precompute/"
    ):
        if isinstance(cache, str):
            cache = Path(cache)
        cache_path = cache / f"{index}.json"
        with open(cache_path, mode="r", encoding="utf-8") as file:
            raw = file.read()
            glacier_index = dict(json.loads(raw))

        return glacier_index

    def get_cached_outline_data(self, cache_path: Path) -> gpd.GeoDataFrame:
        """Get glacier outlines.

        This is identical to ``gdir.read_shapefile``, so the CRS should
        later be converted to EPSG:4236"""
        try:
            glacier_outlines = gpd.read_feather(cache_path / "outlines.shp")
        except FileNotFoundError:
            return None

        return glacier_outlines

    def get_cached_region_outlines(
        self,
        region_id: int,
        file_name="glacier_outlines.shp",
    ) -> gpd.GeoDataFrame:
        """Get subregion domain outlines.

        Parameters
        ----------
        region_id : int
            O1 region ID number.
        file_name : str
            Shapefile name.

        """
        # region_id = int(rgi_id[6:8])
        regions = {"Central Europe": 11, "Iceland": 6}
        try:
            shapefile_path = Path(
                self.cache_path
                / f"RGI60-{str(regions[region_id]).zfill(2)}/{file_name}"
            )
            shapefile = gpd.read_feather(shapefile_path)
        except FileNotFoundError:
            return None

        return shapefile


# Singleton instance
data_cache = GlacierDataCache()
print("✓ Data layer initialized")

## 2. Plotting Layer

In [ ]:
import logging


class GlacierPlotter:
    """Create plots."""

    def __init__(self):
        # Create plot objects ONCE
        self.plot_cryo = dtcg_plotting.BokehSynthetic()
        self.plot_map = dtcg_plotting.BokehMapOutlines()
        self.plot_graph = dtcg_plotting.BokehGraph()

    def create_l2_plots(
        self, data: dict, year: int, model_name: str = "DailyTIModel"
    ) -> tuple:
        """Create L2 Bokeh plots from data."""
        gdir = data["gdir"]
        datacube = data.get("eolis", None)
        smb = data["smb"]
        runoff_data = data["runoff_data"]["Daily_Hugonnet_2000_2020"]
        figures = []

        fig_daily_mb = self.plot_cryo.plot_mb_comparison(
            smb=smb,
            ref_year=year,
            datacube=datacube,
            gdir=gdir,
            cumulative=False,
            model_name=model_name,
        )
        fig_cumulative_mb = self.plot_cryo.plot_mb_comparison(
            smb=smb,
            ref_year=year,
            datacube=datacube,
            gdir=gdir,
            cumulative=True,
            model_name=model_name,
        )

        fig_monthly_runoff = self.plot_graph.plot_runoff_timeseries(
            runoff=runoff_data["monthly_runoff"], ref_year=year, nyears=27
        )
        fig_runoff_cumulative = self.plot_graph.plot_runoff_timeseries(
            runoff=runoff_data["monthly_runoff"],
            ref_year=year,
            cumulative=True,
            nyears=27,
        )
        if datacube is not None:
            model = "OGGM + CryoSat"
        else:
            model = "OGGM"

        figures = [
            fig_daily_mb.opts(title=f"Specific Mass Balance ({model})"),
            fig_cumulative_mb.opts(title=f"Cumulative Specific Mass Balance ({model})"),
            fig_monthly_runoff,
            fig_runoff_cumulative,
        ]

        return figures

    def create_l1_plots(self, data: dict, year: int) -> tuple:
        """Create L1 Bokeh plots from data."""
        gdir = data["gdir"]
        datacube = data.get("eolis", None)

        figures = []
        if datacube is not None:
            fig_eo_elevation = self.plot_cryo.plot_eolis_timeseries(
                datacube=datacube,
                mass_balance=True,
                glacier_area=gdir.get("rgi_area_km2", None),
            ).opts(title="Monthly Cumulative Specific Mass Balance (CryoSat)")

            fig_eo_smb = self.plot_cryo.plot_eolis_smb(
                datacube=datacube,
                ref_year=year,
                years=None,
                cumulative=False,
                glacier_area=gdir.get("rgi_area_km2", None),
            ).opts(title="Cumulative Specific Mass Balance (CryoSat)")
            figures = [fig_eo_elevation, fig_eo_smb]

        return figures

    def plot_selection_map(self, shapefile, rgi_id) -> hv.Layout:
        """Plot map showing the selected glacier.

        Parameters
        ----------
        data : dict
            Contains glacier data, shapefile, and optionally runoff
            data and observations.
        glacier_name : str, optional
            Name of glacier in subregion. Default empty string.

        Returns
        -------
        hv.Layout
            Dashboard showing a map of the subregion and runoff data.
        """
        try:
            fig_glacier_highlight = self.plot_map.plot_region_with_glacier(
                shapefile=shapefile, rgi_id=rgi_id
            ).opts(xlabel="", ylabel="", xaxis=None, yaxis=None, scalebar=True)
            return fig_glacier_highlight
        except:
            return pn.pane.Markdown(f"""#Test""")


plotter = GlacierPlotter()
print("Plotting layer initialised")

## 3. Streaming layer

In [ ]:
from dtcg.api.external.call import StreamDatacube

streamer = StreamDatacube(
    server="https://cluster.klima.uni-bremen.de/~dtcg/datacubes_case_study_regions/v2026.2/L1_and_L2/"
)
print("Data cube streaming initialised")

## 4. Dashboard Controller

In [ ]:
class Dashboard:
    """Arranges UI and updates data in place."""

    def __init__(self):
        self.plots_oggm_container = pn.Column(
            pn.pane.Markdown("### Select a glacier to view data"),
            sizing_mode="stretch_width",
        )
        self.plots_cryosat_container = pn.Column(
            pn.pane.Markdown("### No CryoSat data available.", name="CryoSat Data"),
            sizing_mode="stretch_width",
        )
        self.map = pn.FlexBox(pn.pane.Markdown(f"""Test init"""))
        self.glacier_info = pn.pane.HTML()

        pn.io.loading.start_loading_spinner(self.plots_oggm_container)
        glacier_index = data_cache.get_glacier_index()
        regions = list(glacier_index.keys())

        self.region_selector = pn.widgets.Select(
            name="Region", options=regions, value=regions[0]
        )
        self.glacier_selector = pn.widgets.Select(
            name="Glacier",
            options=list(glacier_index[regions[0]].keys()),
            value=list(glacier_index[regions[0]].keys())[0],
        )
        self.year_selector = pn.widgets.Select(
            name="Year",
            options=list(range(int(date.today().year) - 1, 1999, -1)),
            value=int(date.today().year) - 1,
        )
        self.model_selector = pn.widgets.Select(
            name="OGGM Model",
            options={
                "Daily": "DailyTIModel",
                "Daily Surface Tracking": "SfcTypeTIModel",
            },
            value="DailyTIModel",
        )

        self._glacier_index = glacier_index
        self._current_data: Optional[dict] = None
        self._current_rgi_id: Optional[str] = None
        self._current_year: Optional[str] = None
        self._current_model: Optional[str] = None
        self._current_shapefile: Optional[hv.Polygons] = None

        # Sidebar menus
        self.region_selector.param.watch(self._on_region_change, ["value"])
        self.glacier_selector.param.watch(self._on_glacier_change, ["value"])
        self.year_selector.param.watch(self._on_year_change, ["value"])
        self.model_selector.param.watch(self._on_model_change, ["value"])
        self.glacier_selector.param.trigger("options", "value")

        self.progress_bar = pn.indicators.Progress(
            name="Retrieving Data...", active=True, value=0
        )
        self.download_button = self.set_download_button()
        pn.io.loading.stop_loading_spinner(self.plots_oggm_container)
        pn.io.loading.stop_loading_spinner(self.plots_cryosat_container)

    def _on_region_change(self, *events):
        """Callback for when the region changes."""
        region = self.region_selector.value
        glaciers = list(self._glacier_index[region].keys())
        self._current_shapefile = data_cache.get_cached_region_outlines(
            region_id=region
        )

        self.glacier_selector.options = glaciers
        # Async causes blank plots if glacier change not triggered.
        self.glacier_selector.value = glaciers[0]

    async def _on_year_change(self, *events):
        """Callback for when the year changes."""
        self._current_year = self.year_selector.value
        try:
            self._update_plots()
        finally:
            pn.io.loading.stop_loading_spinner(self.plots_oggm_container)
            pn.io.loading.stop_loading_spinner(self.plots_cryosat_container)
            self.disable_selectors(disabled=False)

    async def _on_model_change(self, *events):
        """Callback for when the model changes."""
        self._current_model = self.model_selector.value
        try:
            self._update_plots()
        finally:
            pn.io.loading.stop_loading_spinner(self.plots_oggm_container)
            pn.io.loading.stop_loading_spinner(self.plots_cryosat_container)
            self.disable_selectors(disabled=False)

    async def _on_glacier_change(self, *events):
        """Callback for when the glacier changes."""
        glacier = self.glacier_selector.value
        region = self.region_selector.value
        year = self.year_selector.value
        model = self.model_selector.value

        # Get RGI ID
        rgi_id = self._glacier_index[region][glacier]

        # Show loading indicator
        start_time = time.time()

        try:
            # Load data asynchronously
            data = await data_cache.get_glacier_data(rgi_id)

            # Store current data
            self._current_data = data
            self._current_rgi_id = rgi_id
            self._current_year = year
            self._current_model = model

            self.map = self.set_map()
            await asyncio.to_thread(self._update_plots())

            print("Setting map")
            self.download_button = self.set_download_button()

            elapsed = time.time() - start_time
            print(f"✓ Done in {elapsed:.2f}s")
        finally:
            # Hide loading indicator
            pn.io.loading.stop_loading_spinner(self.plots_oggm_container)
            pn.io.loading.stop_loading_spinner(self.plots_cryosat_container)
            self.disable_selectors(disabled=False)

    def _update_plots(self):
        glacier = self.glacier_selector.value
        model = self.model_selector.value

        figures_l2 = plotter.create_l2_plots(
            data=self._current_data, year=self._current_year, model_name=model
        )

        self.plots_oggm_container.objects = [
            pn.pane.Markdown(f"### {glacier} ({self._current_year})"),
            *figures_l2,
        ]
        self.glacier_info.object = self.set_details(self._current_data)

        figures_l1 = plotter.create_l1_plots(
            data=self._current_data, year=self._current_year
        )
        if figures_l1:
            self.plots_cryosat_container.objects = [
                pn.pane.Markdown(f"### {glacier} ({self._current_year})"),
                *figures_l1,
            ]

    def set_map(self):
        """Set map selection pane."""
        # self.map.objects = [pn.pane.Markdown(f"""# Test""")]
        # pn.io.loading.start_loading_indicator(self.map)
        try:
            if self._current_shapefile is not None:
                glacier_map = plotter.plot_selection_map(
                    shapefile=self._current_shapefile, rgi_id=self.rgi_id
                ).opts(max_width=250)
                self.map.objects = [glacier_map]
            else:
                self.map.objects = [pn.pane.Markdown(f"""No shapefile""")]
        except:
            self.map.objects = [pn.pane.Markdown(f"""Map not loaded""")]
        # finally:
        #     pn.io.loading.stop_loading_indicator(self.map)
        return self.map

    def set_details(self, data):
        """Set glacier details pane."""

        if data is not None:
            details = self.get_outline_details(polygon=data["outlines"].iloc[0])
            table = ""
            for k, v in details.items():
                if isinstance(v["value"], float):
                    value = f"{v['value']:.2f}"
                else:
                    value = v["value"]
                table_row = (
                    f"<tr><th>{k}</th><td>{' '.join((f'{value}', v['unit']))}</td></tr>"
                )
                table = f"{table}{table_row}"

            details = (
                f"<hr></hr><h2>Glacier Details</h2><table>{table}</table><hr></hr>"
            )
        return details

    def get_outline_details(self, polygon) -> dict:
        outline_details = {
            "Name": {"value": polygon["Name"], "unit": ""},
            "RGI ID": {"value": polygon["RGIId"], "unit": ""},
            "GLIMS ID": {"value": polygon["GLIMSId"], "unit": ""},
            "Area": {"value": float(polygon["Area"]), "unit": "km²"},
            "Max Elevation": {"value": polygon["Zmax"], "unit": "m"},
            "Min Elevation": {"value": polygon["Zmin"], "unit": "m"},
            "Latitude": {"value": float(polygon["CenLat"]), "unit": "°N"},
            "Longitude": {"value": float(polygon["CenLon"]), "unit": "°E"},
            "Outline Date": {
                "value": self.get_outline_source_date(polygon),
                "unit": "",
            },
        }
        return outline_details

    def get_outline_source_date(self, glacier_data: gpd.GeoDataFrame) -> int:
        """Get the date for an outline's source.

        Parameters
        ----------
        glacier_data : gpd.GeoDataFrame
            Outline data for a glacier. Must conform to
            `RGI60 specifications <https://www.glims.org/RGI/00_rgi60_TechnicalNote.pdf>`__.

        Returns
        -------
        int
            The year the outline's source data was published.
        """
        outline_date = glacier_data.get("EndDate", "-9999999")
        if outline_date == "-9999999":
            outline_date = glacier_data.get("BgnDate", "-9999999")
        outline_date = int(outline_date[:4])

        return outline_date

    async def get_zipped_datacube(
        self, rgi_id, zip_path=Path("./static/data/zarr_data/")
    ) -> Path:
        path = streamer.get_zip_path(zip_path=zip_path, rgi_id=self._current_rgi_id)
        self.download_button.disabled = True
        self.disable_selectors(disabled=True)
        self.progress_bar.value = -1
        try:
            path = await asyncio.to_thread(
                streamer.zip_datacube(zip_path=zip_path, rgi_id=self._current_rgi_id)
            )
        except FileNotFoundError as e:
            pn.state.notifications.position = "bottom-left"
            pn.state.notifications.error(
                "No datacube available for this glacier.", duration=3000
            )
            path = ""
        finally:
            print(f"Datacube path: {path}")
            self.download_button.disabled = False
            self.disable_selectors(disabled=False)
            self.progress_bar.value = 0
            return path  # avoid unbound local errors and other such things

    def get_filename(self):
        filename = f"{self._current_rgi_id}.zarr.zip"
        return filename

    def set_download_button(self):
        self.download_button = pn.widgets.FileDownload(
            callback=pn.bind(self.get_zipped_datacube, rgi_id=self._current_rgi_id),
            # filename=f"{self._current_rgi_id}.zarr.zip",
            label="Download Datacube",
        )

        return self.download_button

    def display_loading_indicator(self, glacier, year, model, *events):
        pn.io.loading.start_loading_spinner(self.plots_oggm_container)
        pn.io.loading.start_loading_spinner(self.plots_cryosat_container)
        self.disable_selectors(disabled=True)

    def disable_selectors(self, disabled=True, *events):
        for selector in [
            self.glacier_selector,
            self.year_selector,
            self.region_selector,
            self.model_selector,
        ]:
            selector.disabled = disabled

    def build_dashboard(self) -> pn.template.MaterialTemplate:
        """Compose the dashboard."""

        sidebar = pn.Column(
            pn.pane.Markdown("### Glacier Selection"),
            self.region_selector,
            self.glacier_selector,
            self.year_selector,
            self.model_selector,
            self.map,
            self.glacier_info,
            self.progress_bar,
            self.download_button,
            sizing_mode="stretch_width",
        )

        main = pn.Column(
            pn.Tabs(
                (
                    "Model (OGGM)",
                    pn.Column(
                        pn.param.ParamFunction(
                            pn.bind(
                                self.display_loading_indicator,
                                glacier=self.glacier_selector,
                                year=self.year_selector,
                                model=self.model_selector,
                            ),
                            loading_indicator=False,
                        ),
                        self.plots_oggm_container,
                    ),
                ),
                ("EO (Cryosat)", self.plots_cryosat_container),
                styles={
                    "flex": "0 0 auto",
                    "align-items": "stretch",
                    "align-content": "stretch",
                    "flex-wrap": "nowrap",
                },
                sizing_mode="stretch_width",
            ),
            styles={
                "flex": "1 1 auto",
                "align-items": "stretch",
                "align-content": "stretch",
                "flex-wrap": "nowrap",
            },
            sizing_mode="stretch_width",
        )

        template = pn.template.MaterialTemplate(
            title="Alpine and Icelandic Glacier Dashboard",
            busy_indicator=pn.indicators.LoadingSpinner(size=40),
            sidebar=sidebar,
            main=main,
            logo="./static/img/dtc_logo_inv_min.png",
            sidebar_width=250,
        )

        return template


dashboard = Dashboard()
print("Dashboard controller initialized")

## 5. Serve Dashboard

In [ ]:
template = dashboard.build_dashboard()
template.servable()